In [11]:
# Setup automático: asegurar que el proyecto esté en sys.path y verificar imports
import sys, os
print('Current cwd:', os.getcwd())
# Añadir candidatos de raíz del proyecto
candidates = [
    os.path.abspath(os.path.join(os.getcwd(), '..')),
    os.path.abspath(os.path.join(os.getcwd(), '..', '..')),
    os.path.abspath(os.getcwd())
]
for p in candidates:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
        print('Añadido a sys.path:', p)

# Verificar imports críticos
missing = []
try:
    import pipeline
    print('Import pipeline: OK ->', getattr(pipeline, '__file__', '?'))
except Exception as e:
    print('Import pipeline: ERROR ->', type(e).__name__, e)
    missing.append('pipeline')
try:
    import ipywidgets
except Exception as e:
    print('Import ipywidgets: ERROR ->', type(e).__name__, e)
    missing.append('ipywidgets')

if missing:
    print('Faltan módulos o imports:', missing)
    print('Ejecuta la celda de instalación o instala los paquetes necesarios.')


Current cwd: c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\notebooks
Import pipeline: OK -> c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\pipeline.py


In [12]:
# SELECTOR DE ARCHIVOS .CDF
# Selecciona los archivos → pulsa "Confirmar" → el pipeline arranca en la celda siguiente
from IPython.display import display, HTML, clear_output
from ipywidgets import SelectMultiple, Button, VBox, HBox, Label, Output
import IPython, pipeline, config
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

original_list_cdf  = pipeline.list_cdf_files
selected_cdf_paths = []
out_run = Output()

# ── Widgets ──────────────────────────────────────────────────────────────────
existing_list = SelectMultiple(rows=8, description='', layout={'width': '340px'})
selected_list = SelectMultiple(rows=8, description='', layout={'width': '340px'})
browse_btn  = Button(description='+ Browse archivos',      button_style='info',    layout={'width': '170px'})
add_btn     = Button(description='Añadir →',               button_style='primary', layout={'width': '110px'})
remove_btn  = Button(description='← Quitar',               button_style='warning', layout={'width': '110px'})
confirm_btn = Button(description='▶ Confirmar y analizar', button_style='success', layout={'width': '200px'})
reset_btn   = Button(description='↺ Reiniciar',            button_style='',        layout={'width': '120px'})
status_lbl  = Label(value='')

def _refresh_existing():
    try:    paths = list(original_list_cdf())
    except: paths = []
    existing_list.options = [(p.name, str(p)) for p in paths]
_refresh_existing()

def _on_browse(b):
    try:
        root = tk.Tk(); root.withdraw(); root.wm_attributes('-topmost', True)
        paths = filedialog.askopenfilenames(
            title='Selecciona archivos .cdf',
            filetypes=[('CDF files', '*.cdf *.CDF'), ('Todos', '*.*')])
        root.destroy()
    except Exception as e:
        status_lbl.value = f'Error explorador: {e}'; return
    curr = dict(existing_list.options)
    for p in paths:
        src = Path(p)
        if src.suffix.upper() == '.CDF' and str(src) not in curr.values():
            curr[src.name] = str(src)
    existing_list.options = list(curr.items())

def _on_add(b):
    vals = list(existing_list.value)
    if not vals: status_lbl.value = 'Selecciona en Existing primero.'; return
    curr = dict(selected_list.options)
    for v in vals:
        if v not in curr.values(): curr[Path(v).name] = v
    selected_list.options = list(curr.items())
    status_lbl.value = f'{len(selected_list.options)} archivo(s) en Selected.'

def _on_remove(b):
    to_remove = set(selected_list.value)
    selected_list.options = [(n, v) for n, v in selected_list.options if v not in to_remove]

def _on_reset(b):
    global selected_cdf_paths
    selected_cdf_paths = []
    pipeline.list_cdf_files = original_list_cdf
    IPython.get_ipython().user_ns.pop('selected_cdf_paths', None)
    selected_list.options = []
    for w in [browse_btn, add_btn, confirm_btn, existing_list, selected_list]: w.disabled = False
    with out_run: clear_output()
    _refresh_existing()
    status_lbl.value = 'Reiniciado.'

def _on_confirm(b):
    global selected_cdf_paths
    opts = list(selected_list.options)
    if not opts: status_lbl.value = '⚠ Añade archivos a Selected primero.'; return

    selected_cdf_paths = [Path(v) for _, v in opts]
    pipeline.list_cdf_files = lambda data_dir=None: selected_cdf_paths
    # Exponer en el namespace del kernel para que la celda siguiente lo lea
    IPython.get_ipython().user_ns['selected_cdf_paths'] = selected_cdf_paths

    for w in [browse_btn, add_btn, confirm_btn, existing_list, selected_list]: w.disabled = True
    status_lbl.value = f'⏳ {len(selected_cdf_paths)} archivo(s) — ejecutando pipeline…'

    with out_run:
        clear_output(wait=True)
        ok = _exec_cell('3958fffb')
        if ok:
            status_lbl.value = f'✔ Pipeline completado — {len(selected_cdf_paths)} muestra(s).'
        else:
            status_lbl.value = '❌ Error en el pipeline (ver output).'

def _exec_cell(cell_id):
    # Lee el notebook y ejecuta la celda con el id dado via ip.run_cell()
    import json, os
    ip = IPython.get_ipython()
    ns = ip.user_ns

    # VS Code inyecta __vsc_ipynb_file__ con la ruta del notebook
    nb_path = ns.get('__vsc_ipynb_file__', None)
    if not nb_path:
        hits = sorted(Path(os.getcwd()).glob('*.ipynb'))
        nb_path = str(hits[0]) if hits else None

    if not nb_path or not Path(nb_path).exists():
        print(f'⚠ No se localizó el notebook.')
        print(f'  Ejecuta manualmente la celda del pipeline (id: {cell_id}).')
        status_lbl.value = '⚠ Ejecuta la celda del pipeline manualmente.'
        return False

    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    for cell in nb['cells']:
        if cell.get('id') == cell_id and cell['cell_type'] == 'code':
            src = ''.join(cell['source']).strip()
            if not src:
                return True
            result = ip.run_cell(src)
            return not (result.error_before_exec or result.error_in_exec)

    print(f'⚠ Celda "{cell_id}" no encontrada en el notebook.')
    return False

browse_btn.on_click(_on_browse); add_btn.on_click(_on_add); remove_btn.on_click(_on_remove)
confirm_btn.on_click(_on_confirm); reset_btn.on_click(_on_reset)

display(HTML('<h3>📂 Seleccionar archivos .cdf para el análisis</h3>'))
display(browse_btn)
display(HBox([
    VBox([Label('📂 Existing (disponibles)'), existing_list]),
    VBox([add_btn, remove_btn], layout={'justify_content': 'center', 'margin': '40px 12px'}),
    VBox([Label('✅ Selected (a analizar)'), selected_list]),
]))
display(HBox([confirm_btn, reset_btn]))
display(status_lbl)
display(HTML('<hr style="margin-top:12px"><b>Output del pipeline:</b>'))
display(out_run)


Button(button_style='info', description='+ Browse archivos', layout=Layout(width='170px'), style=ButtonStyle()…

Label(value='')

Output()

# AutoGCMS Pipeline

Pipeline automatizado para procesamiento de datos GC-MS.

**Flujo:**
1. **Exploración** (una muestra): preprocesado, detección de picos, espectros, identificación NIST
2. **Pipeline completo** (todas las muestras): módulos 1-6 automáticos → `final_report/`

**Requisito:** Exportar MAINLIB desde NIST MS Search a formato .msp y configurar la ruta en `config.py` (`NIST_MSP_PATH`).

---

## 1. Seleccionar archivo .cdf

Cambia `FILE_IDX` para procesar otra muestra.

In [13]:
import sys
from pathlib import Path

# Añadir el directorio padre al path de Python para encontrar los módulos
notebook_dir = Path.cwd()
parent_dir = notebook_dir.parent
sys.path.insert(0, str(parent_dir))

import matplotlib.pyplot as plt
import config
from pipeline import list_cdf_files
from preprocessing import preprocess
from peak_detection import find_and_filter_peaks
from spectra import extract_all_spectra, export_msp
from visualization import plot_tic_preprocessing, plot_peaks, plot_peak_quality, plot_spectrum

print("✓ Importaciones cargadas correctamente")

✓ Importaciones cargadas correctamente


In [14]:
cdf_files = list_cdf_files()
print(f"Archivos .cdf disponibles:\n")
for i, f in enumerate(cdf_files):
    print(f"  [{i}] {f.name}")

# === SELECCIONAR AQUI ===
FILE_IDX = 0
# ========================

test_file = cdf_files[FILE_IDX]
print(f"\nSeleccionado: {test_file.name}")

Archivos .cdf disponibles:

  [0] Exp_1.CDF
  [1] Exp_2.CDF
  [2] Exp_3.CDF

Seleccionado: Exp_1.CDF


---
## 2. Preprocesado del cromatograma (Modulo 1)

Lee el archivo .cdf, aplica suavizado Savitzky-Golay y correccion de baseline.

In [16]:
data = preprocess(test_file)
print(f"Muestra: {data.name}")
print(f"Scans: {data.n_scans}")
print(f"Tiempo: {data.scan_times_min[0]:.2f} - {data.scan_times_min[-1]:.2f} min")
print(f"Rango m/z: {data.mass_range[0]:.1f} - {data.mass_range[1]:.1f}")

Muestra: Exp_1
Scans: 16478
Tiempo: 0.09 - 59.22 min
Rango m/z: 50.1 - 347.1


In [17]:
from visualization import plot_tic_preprocessing
fig = plot_tic_preprocessing(data)
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12924\3664819497.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Deteccion de picos (Modulo 2)

In [18]:
from peak_detection import find_and_filter_peaks

peaks = find_and_filter_peaks(data)
print(f"Picos detectados: {peaks.n_peaks}")
print(f"  (peak_id = ranking por intensidad, scan_idx = indice real en TIC)")
print(f"\nTop 20 picos por intensidad:")
peaks.table.head(20)[['peak_id', 'scan_idx', 'rt_min', 'intensity', 'snr', 'width_sec']]

Picos detectados: 46
  (peak_id = ranking por intensidad, scan_idx = indice real en TIC)

Top 20 picos por intensidad:


,peak_id,scan_idx,rt_min,intensity,snr,width_sec
0,1,764,2.833033,294482.314324,3.219140,1.953976
1,2,796,2.947867,291895.407697,3.459176,2.009063
2,3,4743,17.112233,194643.563975,3.068972,3.773443
3,4,3793,13.703017,179929.517861,3.015983,8.256867
4,5,2897,10.487600,173002.356396,6.045763,12.988241
5,6,1198,4.390500,147579.610285,3.154576,2.994737
6,7,1739,6.331967,129942.844808,3.688302,2.560030
7,8,4465,16.114583,114944.446824,3.023337,4.126547
8,9,656,2.445467,112867.395786,3.009105,5.774109
9,10,5449,19.645817,112847.240886,4.084203,4.305852


In [19]:
from visualization import plot_peaks, plot_peak_quality

fig = plot_peaks(data, peaks)
plt.show()

fig = plot_peak_quality(peaks)
plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12924\3170286212.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12924\3170286212.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Extraccion de espectros de masas (Modulo 3)

In [20]:
from spectra import extract_all_spectra, export_msp

spectra = extract_all_spectra(data, peaks)
n_coeluted = sum(1 for s in spectra if s.is_coeluted)
n_subpeaks = sum(1 for s in spectra if s.subpeak > 0)
print(f"Espectros extraidos: {len(spectra)}")
print(f"  Picos simples: {len(spectra) - n_subpeaks}")
print(f"  Picos co-eluidos (Gaussiana BIC): {n_coeluted}")
print(f"  Subpicos generados: {n_subpeaks}")

# Exportar a .msp
msp_path = config.OUTPUT_DIR / f"{data.name}_spectra.msp"
export_msp(spectra, msp_path, sample_name=data.name)
print(f"\nExportados a: {msp_path}")

Espectros extraidos: 89
  Picos simples: 3
  Picos co-eluidos (Gaussiana BIC): 86
  Subpicos generados: 86

Exportados a: c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\output\Exp_1_spectra.msp


In [21]:
from visualization import plot_spectrum

# Espectro del pico mas intenso (subpeak=0 = pico simple)
simple_spectra = [s for s in spectra if s.subpeak == 0]
sp = simple_spectra[0] if simple_spectra else spectra[0]

sub_label = f" (subpeak {sp.subpeak})" if sp.subpeak > 0 else ""
fig = plot_spectrum(
    sp.mz, sp.intensities,
    title=f"Espectro de masas -- Pico {sp.peak_id}{sub_label} (RT={sp.rt_min:.2f} min)"
)
plt.show()

print(f"\nFragmentos principales (m/z : intensidad):")
for idx in sp.intensities.argsort()[-10:][::-1]:
    print(f"  m/z {sp.mz[idx]:6.1f} : {sp.intensities[idx]:10.0f}")


Fragmentos principales (m/z : intensidad):
  m/z   57.1 :        523
  m/z   57.0 :        258
  m/z   85.1 :        195
  m/z   70.9 :        182
  m/z   61.1 :        164
  m/z   70.0 :        151
  m/z   56.0 :        138
  m/z   56.2 :        126
  m/z   83.1 :        124
  m/z   71.0 :        122


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12924\3670317339.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Identificacion de compuestos (Modulo 4) -- Libreria NIST

Compara cada espectro contra la libreria NIST (cargada en el setup) usando similitud coseno.  
Devuelve los top N candidatos por espectro con score >= umbral.

In [22]:
from nist_search import load_nist_library, _finalize_spectrum
from nist_binary_reader import try_load_nist_mainlib_binary
from pathlib import Path
import numpy as np

print(f"\n{'='*100}")
print(f"CARGANDO LIBRERÍA NIST MAINLIB")
print(f"{'='*100}\n")

nist_dir = Path(r"C:\Users\LENOVO\OneDrive\Documentos\MATLAB\MSSEARCH")

# ——— INTENTAR 1: LEER MAINLIB BINARIO DIRECTAMENTE (Sin exportar!) ———
print("⚙️  Intento 1: Leer MAINLIB binario directamente...")
library = try_load_nist_mainlib_binary(nist_dir, max_attempts=5000)

if not library:
    print("❌ No se pudo leer MAINLIB binario\n")
    
    # ——— INTENTAR 2: .msp con contenido ———
    print("⚙️  Intento 2: Buscar archivos .msp locales...")
    
    all_msp_files = list(nist_dir.rglob("*.msp")) + list(nist_dir.rglob("*.MSP"))
    msp_with_content = [f for f in all_msp_files if f.stat().st_size > 100_000]
    
    if msp_with_content:
        msp_by_size = sorted(msp_with_content, key=lambda f: f.stat().st_size, reverse=True)
        best_msp = msp_by_size[0]
        print(f"✅ Encontrado: {best_msp.name} ({best_msp.stat().st_size / (1024*1024):.1f} MB)")
        library = load_nist_library(str(best_msp))
    else:
        print("❌ No hay .msp con contenido\n")
        
        # ——— INTENTO 3: Fallback a 17 compuestos default ———
        print("⚙️  Intento 3: Usar librería default (17 compuestos)...")
        library = load_nist_library()

# ——— VALIDAR Y NORMALIZAR ———
if not library:
    library = []

# Normalizar espectros que vengan del lector binario (add mz/intensities si faltan)
for spec in library:
    if "mz" not in spec or "intensities" not in spec:
        spec["mz"] = np.array([], dtype=np.float32)
        spec["intensities"] = np.array([], dtype=np.float32)

# ——— RESULTADO FINAL ———
print(f"\n{'='*100}")
print(f"✅ LIBRERÍA CARGADA EXITOSAMENTE")
print(f"{'='*100}")
print(f"📊 Compuestos de referencia: {len(library)}")
print(f"🔬 Espectros a buscar: {len(spectra)}")
print(f"⚙️  Umbral coseno: {config.COSINE_THRESHOLD}")
print(f"🏆 Top candidatos: {config.NIST_TOP_N}")

if len(library) > 0:
    # Mostrar información de primeros 3
    print(f"\nPrimeros 3 compuestos en librería:")
    for i, comp in enumerate(library[:3]):
        mz_count = len(comp.get("mz", []))
        formula = comp.get("formula", "")
        print(f"  {i+1}. {comp.get('name', 'Unknown'):40s} | m/z picos: {mz_count:3d} | {formula}")
else:
    print("\n⚠️  librería vacía!")

Archivos MAINLIB.db o MAINLIB.in6 no encontrados



CARGANDO LIBRERÍA NIST MAINLIB

⚙️  Intento 1: Leer MAINLIB binario directamente...
❌ No se pudo leer MAINLIB binario

⚙️  Intento 2: Buscar archivos .msp locales...
❌ No hay .msp con contenido

⚙️  Intento 3: Usar librería default (17 compuestos)...

✅ LIBRERÍA CARGADA EXITOSAMENTE
📊 Compuestos de referencia: 17
🔬 Espectros a buscar: 89
⚙️  Umbral coseno: 0.5
🏆 Top candidatos: 5

Primeros 3 compuestos en librería:
  1. Cocaine                                  | m/z picos:  87 | C17H21NO4
  2. Cortisone                                | m/z picos: 131 | C21H28O5
  3. Cocaine                                  | m/z picos: 170 | C17H21NO4


In [23]:
# DEBUG: Ver TODOS los compuestos en la librería
print(f"\n{'='*100}")
print(f"DEBUG: TODOS LOS COMPUESTOS EN LA LIBRERÍA NIST")
print(f"{'='*100}\n")

for i, comp in enumerate(library):
    name = comp.get('name', 'NO NAME')
    formula = comp.get('formula', '')
    cas = comp.get('cas', '')
    
    # Detectar si es un compuesto real o de demostración
    is_demo = 'Scan' in name or 'DEMO' in name
    marker = "⚠️ DEMO" if is_demo else "✅ REAL"
    
    print(f"{i:2d}. [{marker}] {name}")
    if formula:
        print(f"    Formula: {formula}, CAS: {cas}")



DEBUG: TODOS LOS COMPUESTOS EN LA LIBRERÍA NIST

 0. [✅ REAL] Cocaine
    Formula: C17H21NO4, CAS: 50-36-2;  EPA#: 113834
 1. [✅ REAL] Cortisone
    Formula: C21H28O5, CAS: 
 2. [✅ REAL] Cocaine
    Formula: C17H21NO4, CAS: 
 3. [✅ REAL] Benzoic acid, 2-(acetyloxy)-
    Formula: C9H8O4, CAS: 
 4. [✅ REAL] Heroin
    Formula: C21H23NO5, CAS: 
 5. [✅ REAL] Benzene
    Formula: C6H6, CAS: 
 6. [✅ REAL] Benzene, nitro- (1)
    Formula: C6H5NO2, CAS: 
 7. [✅ REAL] Benzene, nitro- (2)
    Formula: C6H5NO2, CAS: 
 8. [✅ REAL] Benzene, nitro- (3)
    Formula: C6H5NO2, CAS: 
 9. [✅ REAL] Benzene, nitro- (4)
    Formula: C6H5NO2, CAS: 
10. [✅ REAL] Benzene, nitro- (5)
    Formula: C6H5NO2, CAS: 
11. [✅ REAL] Benzene, nitro- (6)
    Formula: C6H5NO2, CAS: 
12. [✅ REAL] Benzene, nitro- (7)
    Formula: C6H5NO2, CAS: 
13. [✅ REAL] Benzene, nitro- (8)
    Formula: C6H5NO2, CAS: 98953;  EPA# 56474; Source: ERL; QI: 571
14. [✅ REAL] Benzene, nitro- (9)
    Formula: C6H5NO2, CAS: 
15. [⚠️ DEMO] Scan 2

In [24]:
# Matching contra libreria NIST usando similitud coseno
from nist_search import match_all_spectra
import pandas as pd

print(f"\n{'='*100}")
print(f"IDENTIFICACIÓN DE COMPUESTOS CONTRA LIBRERÍA NIST")
print(f"{'='*100}")
print(f"Librería: {len(library)} compuestos (15 reales + 2 de demo)")
print(f"Espectros a buscar: {len(spectra)}")
print(f"Umbral de similitud: 0.3\n")

threshold_for_matching = 0.3

matches = match_all_spectra(
    spectra, library,
    top_n=5,
    threshold=threshold_for_matching,
    sample_name=data.name,
)

# Construir tabla con TODOS los matches
results = []
for spectrum_idx, match_result in enumerate(matches):
    spectrum = spectra[spectrum_idx]
    
    if not match_result.candidates:
        # Sin matches
        results.append({
            'peak_id': spectrum.peak_id,
            'subpeak': spectrum.subpeak,
            'rt_min': spectrum.rt_min,
            'rank': 1,
            'compound_name': 'No match',
            'formula': '',
            'cas': '',
            'match_score': 0.0,
            'n_matched_peaks': 0,
            'library_type': 'NONE'
        })
    else:
        # Con matches
        for rank, candidate in enumerate(match_result.candidates, 1):
            lib_idx = candidate.get('library_idx', None)
            score = candidate.get('score', 0.0)
            n_peaks = candidate.get('n_matched_peaks', 0)
            
            # Obtener datos del compuesto de la librería
            if lib_idx is not None and 0 <= lib_idx < len(library):
                lib_compound = library[lib_idx]
                compound_name = lib_compound.get('name', 'Unknown')
                formula = lib_compound.get('formula', '')
                cas = lib_compound.get('cas', '')
                
                # Identificar si es compuesto real o demo
                if 'Scan' in compound_name or 'DEMO' in compound_name:
                    library_type = 'DEMO'
                else:
                    library_type = 'REAL'
            else:
                compound_name = 'Unknown'
                formula = ''
                cas = ''
                library_type = 'UNKNOWN'
            
            results.append({
                'peak_id': spectrum.peak_id,
                'subpeak': spectrum.subpeak,
                'rt_min': spectrum.rt_min,
                'rank': rank,
                'compound_name': compound_name,
                'formula': formula,
                'cas': cas,
                'match_score': score,
                'n_matched_peaks': n_peaks,
                'library_type': library_type
            })

id_table = pd.DataFrame(results)

# —— ESTADÍSTICAS ——
rank1_all = id_table[id_table['rank'] == 1]
n_real = len(rank1_all[rank1_all['library_type'] == 'REAL'])
n_demo = len(rank1_all[rank1_all['library_type'] == 'DEMO'])
n_no_match = len(rank1_all[rank1_all['compound_name'] == 'No match'])

print(f"{'='*100}")
print(f"RESULTADOS - TODAS LAS IDENTIFICACIONES")
print(f"{'='*100}")
print(f"✅ Compuestos REALES identificados: {n_real}")
print(f"⚠️  Matches a espectros DEMO: {n_demo}")
print(f"❌ Sin match: {n_no_match}")
print(f"📊 Total: {len(rank1_all)} picos\n")

# Guardar tabla
id_csv = config.OUTPUT_DIR / f"{data.name}_identifications.csv"
id_table.to_csv(id_csv, sep=';', encoding='utf-8-sig', index=False)
print(f"📁 Tabla guardada en: {id_csv}\n")

# Mostrar TODOS los picos identificados
cols_display = ['peak_id', 'subpeak', 'rt_min', 'rank', 'compound_name', 'formula',
                'match_score', 'n_matched_peaks', 'library_type']
                
# Ordenar por peak_id y luego por rank
display_table = id_table[id_table['rank'] == 1][cols_display].sort_values('peak_id')

print(f"{'='*100}")
print(f"🔬 TODOS LOS PICOS DETECTADOS Y SUS IDENTIFICACIONES (rank 1)")
print(f"{'='*100}\n")
display(display_table.reset_index(drop=True))


IDENTIFICACIÓN DE COMPUESTOS CONTRA LIBRERÍA NIST
Librería: 17 compuestos (15 reales + 2 de demo)
Espectros a buscar: 89
Umbral de similitud: 0.3

RESULTADOS - TODAS LAS IDENTIFICACIONES
✅ Compuestos REALES identificados: 7
⚠️  Matches a espectros DEMO: 20
❌ Sin match: 62
📊 Total: 89 picos

📁 Tabla guardada en: c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\output\Exp_1_identifications.csv

🔬 TODOS LOS PICOS DETECTADOS Y SUS IDENTIFICACIONES (rank 1)



,peak_id,subpeak,rt_min,rank,compound_name,formula,match_score,n_matched_peaks,library_type
0,1,1,2.833033,1,No match,,0.000000,0,NONE
1,1,2,2.872517,1,No match,,0.000000,0,NONE
2,2,1,2.947867,1,No match,,0.000000,0,NONE
3,2,2,2.969400,1,No match,,0.000000,0,NONE
4,3,1,17.112233,1,Scan No. 519 DEMO,,0.756793,44,DEMO
...,...,...,...,...,...,...,...,...,...
84,44,0,29.912900,1,No match,,0.000000,0,NONE
85,45,1,11.880000,1,Cocaine,C17H21NO4,0.366334,26,REAL
86,45,2,11.887167,1,No match,,0.000000,0,NONE
87,46,1,58.076533,1,Scan 2313 (32.345 min) of MAY041201012.d 10684,,0.397187,20,DEMO


---
## 6. Pipeline completo: TODAS las muestras + Reporte final

Procesa automaticamente **todos** los archivos .cdf del directorio de datos:
- Modulos 1-4 por cada muestra (con libreria NIST)
- Modulo 5: alineamiento de picos entre muestras
- Modulo 6: genera `final_report/` con QC, PCA, heatmap y XLSX multi-hoja

In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# PIPELINE COMPLETO — módulos 1-6 para TODAS las muestras seleccionadas
# Se ejecuta automáticamente al pulsar "▶ Confirmar" en el selector de arriba.
# También se puede ejecutar manualmente (selected_cdf_paths debe estar definido).
# ═══════════════════════════════════════════════════════════════════════════════
import traceback, tempfile, shutil, unicodedata
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import IPython, config
from pathlib import Path
from pipeline import run_single_sample
from matrix_builder import (SampleResult, align_peaks,
                             build_data_matrix, build_feature_metadata,
                             export_xlsx_report)
from qc import (infer_sample_metadata, preprocess_feature_matrix,
                pca_features, plot_pca_features, plot_heatmap_features,
                compute_advanced_qc_summary)

ns = IPython.get_ipython().user_ns

# ── Archivos a analizar ──────────────────────────────────────────────────────
cdf_files = ns.get('selected_cdf_paths', [])
if not cdf_files:
    import pipeline as _pm
    cdf_files = list(_pm.list_cdf_files())

if not cdf_files:
    print('❌ No hay archivos .cdf seleccionados.')
    print('   Usa el selector de arriba y pulsa "▶ Confirmar y analizar".')
else:
    # ── Librería NIST ────────────────────────────────────────────────────────
    library = ns.get('library', None)
    if library is None:
        from nist_search import load_nist_library
        from nist_binary_reader import try_load_nist_mainlib_binary
        print('⚙️  Cargando librería NIST…')
        _nist_dir = Path(r'C:\Users\LENOVO\OneDrive\Documentos\MATLAB\MSSEARCH')
        library = try_load_nist_mainlib_binary(_nist_dir, max_attempts=5000) or []
        if not library:
            library = load_nist_library()
        for spec in library:
            if 'mz' not in spec or 'intensities' not in spec:
                spec['mz'] = np.array([], dtype=np.float32)
                spec['intensities'] = np.array([], dtype=np.float32)
        ns['library'] = library
        print(f'✅ Librería lista: {len(library)} compuestos')

    final_report_dir = config.OUTPUT_DIR / 'final_report'
    final_report_dir.mkdir(parents=True, exist_ok=True)

    sep = '═' * 70
    print(f'\n{sep}')
    print(f'  PIPELINE COMPLETO  —  {len(cdf_files)} muestra(s)')
    print(f'{sep}\n')
    print(f'  Librería NIST : {len(library)} compuestos')
    print(f'  Salida        : {final_report_dir}\n')

    def _ascii_path(path):
        try:
            path.name.encode('ascii'); return path
        except Exception:
            safe = ''.join(c for c in unicodedata.normalize('NFKD', path.name)
                           if ord(c) < 128) or 'sample.cdf'
            tmp = Path(tempfile.gettempdir()) / 'autogcms_tmp'
            tmp.mkdir(parents=True, exist_ok=True)
            dest = tmp / safe
            i = 1
            while dest.exists():
                dest = tmp / f'{dest.stem}_{i}{dest.suffix}'; i += 1
            shutil.copy2(str(path), str(dest))
            print(f'  (copia temporal: {path.name} → {dest.name})')
            return dest

    # ── PASO 1: procesar muestras ────────────────────────────────────────────
    print(f'{"─"*70}')
    print('  PASO 1 · Procesamiento individual (módulos 1-4)')
    print(f'{"─"*70}\n')

    sample_results, all_id_tables, failed = {}, [], []
    for cdf_path in cdf_files:
        temp_used = None
        try:
            call_path = _ascii_path(Path(cdf_path))
            temp_used = call_path if call_path != Path(cdf_path) else None
            print(f'  ⏳ {Path(cdf_path).name} … ', end='', flush=True)
            res = run_single_sample(call_path, library=library,
                                    save_plots=True, save_msp=True)
            if not res:
                raise RuntimeError('run_single_sample devolvió None')
            if isinstance(res, tuple) and len(res) >= 5:
                data, peaks, spectra, matches, id_table = res[:5]
            elif isinstance(res, tuple) and len(res) == 4:
                data, peaks, spectra, matches = res; id_table = pd.DataFrame()
            else:
                raise RuntimeError(f'Formato inesperado: {type(res)}')
            sample_results[data.name] = SampleResult(
                name=data.name, peak_table=peaks.table,
                spectra=spectra, matches=matches)
            all_id_tables.append(id_table)
            rank1 = id_table[id_table['rank'] == 1] if not id_table.empty else pd.DataFrame()
            n_id  = len(rank1[rank1['compound_name'] != 'No match']) if not rank1.empty else 0
            n_pks = len(peaks.table) if hasattr(peaks, 'table') else '?'
            n_sp  = len(spectra)     if spectra is not None else '?'
            print(f'✅  ({n_pks} picos · {n_sp} esp. · {n_id} id.)')
        except Exception as e:
            print(f'❌  {type(e).__name__}: {e}')
            failed.append((cdf_path, traceback.format_exc()))
        finally:
            try:
                if temp_used and Path(temp_used).exists(): Path(temp_used).unlink()
            except Exception: pass

    print(f'\n  ✅ Procesadas: {len(sample_results)}/{len(cdf_files)} muestras')
    if failed:
        log = final_report_dir / 'failed_samples.log'
        with open(log, 'w', encoding='utf-8') as fh:
            for p, tb in failed:
                fh.write(f'{p}\n{tb}\n{"─"*60}\n')
        print(f'  ⚠  Fallos guardados en: {log}')

    if not sample_results:
        print('\n❌ No se procesó ninguna muestra. Pipeline abortado.')
    else:
        # ── PASO 2: alineamiento ─────────────────────────────────────────────
        print(f'\n{"─"*70}')
        print('  PASO 2 · Alineamiento de picos (módulo 5)')
        print(f'{"─"*70}\n')

        aligned   = align_peaks(sample_results, rt_tolerance=config.RT_TOLERANCE_MIN)
        s_names   = list(sample_results.keys())
        X_raw     = build_data_matrix(aligned, s_names, value_type='intensity')
        feat_meta = build_feature_metadata(aligned)
        miss_pct  = X_raw.isna().sum().sum() / X_raw.size * 100 if X_raw.size > 0 else 0
        print(f'  Features alineadas : {len(aligned)}')
        print(f'  Matriz             : {X_raw.shape[0]} muestras × {X_raw.shape[1]} features')
        print(f'  Missing values     : {miss_pct:.1f}%')

        # ── PASO 3: QC + reporte ─────────────────────────────────────────────
        print(f'\n{"─"*70}')
        print('  PASO 3 · QC y reporte final (módulo 6)')
        print(f'{"─"*70}\n')

        try:
            meta = infer_sample_metadata(s_names)
            X_proc, kept = preprocess_feature_matrix(X_raw)

            qc_df = compute_advanced_qc_summary(X_raw, X_proc, meta)
            qc_df.to_excel(final_report_dir / 'qc_summary.xlsx', index=False)
            print('  ✅ qc_summary.xlsx')

            export_xlsx_report(final_report_dir / 'feature_matrix_raw.xlsx', X_raw, feat_meta, meta)
            print('  ✅ feature_matrix_raw.xlsx')

            fm_proc = feat_meta.loc[kept] if hasattr(feat_meta, 'loc') else feat_meta
            export_xlsx_report(final_report_dir / 'feature_matrix_processed.xlsx', X_proc, fm_proc, meta)
            print('  ✅ feature_matrix_processed.xlsx')

            if X_proc.shape[0] >= 2 and X_proc.shape[1] >= 2:
                scores, loadings, ev = pca_features(X_proc)
                fig = plot_pca_features(scores, ev, meta)
                fig.savefig(final_report_dir / 'pca_features.png', dpi=150, bbox_inches='tight')
                plt.close(fig)
                print('  ✅ pca_features.png')
                top_n = min(50, X_proc.shape[1])
                fig = plot_heatmap_features(X_proc, meta, top_n=top_n)
                fig.savefig(final_report_dir / 'heatmap_topN_features.png', dpi=150, bbox_inches='tight')
                plt.close(fig)
                print(f'  ✅ heatmap_topN_features.png  (top {top_n} features)')
            else:
                print('  ⚠  PCA/heatmap omitidos (pocas muestras o features).')

        except Exception as e:
            print(f'\n  ⚠  Error en QC/reporte: {type(e).__name__}: {e}')
            traceback.print_exc()

        print(f'\n{sep}')
        print(f'  PIPELINE COMPLETADO  →  {final_report_dir}')
        print(f'{sep}\n')



══════════════════════════════════════════════════════════════════════
  PIPELINE COMPLETO  —  3 muestra(s)
══════════════════════════════════════════════════════════════════════

  Librería NIST : 17 compuestos
  Salida        : c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\output\final_report

──────────────────────────────────────────────────────────────────────
  PASO 1 · Procesamiento individual (módulos 1-4)
──────────────────────────────────────────────────────────────────────

  ⏳ Exp_1.CDF … 

c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\visualization.py:80: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(16, 6))


✅  (46 picos · 89 esp. · 1 id.)
  ⏳ Exp_2.CDF … ✅  (48 picos · 90 esp. · 3 id.)
  ⏳ Exp_3.CDF … ✅  (57 picos · 109 esp. · 3 id.)

  ✅ Procesadas: 3/3 muestras

──────────────────────────────────────────────────────────────────────
  PASO 2 · Alineamiento de picos (módulo 5)
──────────────────────────────────────────────────────────────────────

  Features alineadas : 82
  Matriz             : 3 muestras × 82 features
  Missing values     : 38.6%

──────────────────────────────────────────────────────────────────────
  PASO 3 · QC y reporte final (módulo 6)
──────────────────────────────────────────────────────────────────────


  ⚠  Error en QC/reporte: TypeError: preprocess_feature_matrix() missing 1 required positional argument: 'sample_metadata'

══════════════════════════════════════════════════════════════════════
  PIPELINE COMPLETADO  →  c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\output\final_report
══════════════════════════════════════════════════════════════════════



Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12924\3281170350.py", line 146, in <module>
    X_proc, kept = preprocess_feature_matrix(X_raw)
                   ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
TypeError: preprocess_feature_matrix() missing 1 required positional argument: 'sample_metadata'


---
## Ajuste de parametros

Para procesar otra muestra individual, vuelve a la seccion 1 y cambia `FILE_IDX`.
La seccion 6 siempre procesa **todas** las muestras y genera el reporte completo.

Si necesitas ajustar la sensibilidad, edita `config.py`:

| Parametro | Efecto al aumentar |
|---|---|
| `PEAK_MIN_PROMINENCE` | Menos picos (solo los mas intensos) |
| `PEAK_MIN_DISTANCE` | Menos picos cercanos entre si |
| `PEAK_SNR_THRESHOLD` | Elimina picos con mas ruido |
| `MAX_PEAKS` | Limita el numero total de picos (None = sin limite) |
| `SAVGOL_WINDOW` | Mas suavizado (puede perder picos estrechos) |
| `BASELINE_LAMBDA` | Baseline mas suave / rigida |
| `APEX_SCANS_AVERAGE` | Mas scans promediados (mas robusto, menos resolucion) |
| `COELUTION_BIC_DELTA` | Menos co-eluciones detectadas (criterio mas estricto) |
| `SPECTRA_AVG_SCANS` | Scans promediados en subpicos co-eluidos |
| `TIC_BIN_SIZE_MIN` | Resolucion del binning para PCA QC |
| `COSINE_THRESHOLD` | Menos identificaciones pero mas fiables |

In [26]:
# DEBUG: listar archivos en final_report_dir
from pathlib import Path
try:
    print('final_report_dir =', final_report_dir)
    files = list(Path(final_report_dir).glob('*'))
    print('Ficheros en final_report_dir:')
    for f in files:
        print(' -', f.name)
except NameError:
    print('final_report_dir no está definido en este scope')


final_report_dir = c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\output\final_report
Ficheros en final_report_dir:
 - failed_samples.log
 - feature_matrix_processed.xlsx
 - feature_matrix_raw.xlsx
 - heatmap_feature_list.xlsx
 - heatmap_topN_features.png
 - pca_explained_variance.csv
 - preprocessing_summary.txt
 - qc_summary.xlsx
 - sample_metadata_used.xlsx


In [27]:
# Mostrar contenido de failed_samples.log si existe
from pathlib import Path
p = Path(final_report_dir) / 'failed_samples.log'
if p.exists():
    print('\n---- failed_samples.log ----')
    print(p.read_text(encoding='utf-8'))
else:
    print('No existe failed_samples.log')



---- failed_samples.log ----
C:\Users\LENOVO\Desktop\TFG\CDF\Niño_2_Hidratada.CDFTFG.CDF
Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_780\1613482860.py", line 61, in <module>
    res = run_single_sample(
        cdf_path, library=library, save_plots=True, save_msp=True
    )
  File "c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\pipeline.py", line 80, in run_single_sample
    data = preprocess(filepath)
  File "c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\preprocessing.py", line 170, in preprocess
    data = load_cdf(filepath)
  File "c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\preprocessing.py", line 61, in load_cdf
    ds = nc.Dataset(str(filepath), "r")
  File "src/netCDF4/_netCDF4.pyx", line 2521, in netCDF4._netCDF4.Dataset.__init__
  File "src/netCDF4/_netCDF4.pyx", line 2158, in netCDF4._netCDF4._ensure_nc_success
FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\LENOVO\\Desktop\\TFG\\CDF\\Niño_2_Hidrata

In [28]:
# Verificar cada cdf en cdf_files y si existe en disco
from pathlib import Path
try:
    print('cdf_files:')
    for p in cdf_files:
        pp = Path(p)
        print(' -', pp, 'exists=', pp.exists())
except NameError:
    print('cdf_files no definido en este scope')


cdf_files:
 - C:\Users\LENOVO\OneDrive\Documentos\Exp_1.CDF exists= True
 - C:\Users\LENOVO\OneDrive\Documentos\Exp_2.CDF exists= True
 - C:\Users\LENOVO\OneDrive\Documentos\Exp_3.CDF exists= True


In [29]:
# Comprobar dependencias Python requeridas y sugerir `pip install` si faltan
import importlib, sys
modules = ['numpy','pandas','netCDF4','seaborn','scipy','sklearn','matplotlib','openpyxl','ipywidgets']
missing = [m for m in modules if importlib.util.find_spec(m) is None]
print('Missing modules:', missing)
if missing:
    print('\nInstalación sugerida (ejecuta en PowerShell):')
    for pkg in missing:
        print(f"{sys.executable} -m pip install {pkg}")

# También comprobar si el módulo local `pipeline` es importable
try:
    import pipeline
    print('\nlocal module pipeline: OK')
except Exception as e:
    print('\nlocal module pipeline: ERROR ->', type(e).__name__, e)


Missing modules: []

local module pipeline: OK


In [30]:
# Intentar añadir el root del proyecto a sys.path y reimportar `pipeline`
import sys, os
cwd = os.getcwd()
candidates = [os.path.abspath(os.path.join(cwd, '..')), os.path.abspath(os.path.join(cwd, '..', '..'))]
added = []
for p in candidates:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
        added.append(p)
print('Paths añadidos a sys.path:', added)
try:
    import pipeline
    print('Import pipeline: OK ->', getattr(pipeline, '__file__', '?'))
except Exception as e:
    print('Import pipeline: ERROR ->', type(e).__name__, e)


Paths añadidos a sys.path: []
Import pipeline: OK -> c:\Users\LENOVO\Desktop\Claude-Proyectos\autogcms\pipeline.py
